# 04-工具调用 - Kimi API

本文档演示 Kimi API 的工具调用(Tool Calling)功能。

In [1]:
from openai import OpenAI
import os
import json
from dotenv import load_dotenv

load_dotenv(dotenv_path='../../.env')

api_key = os.getenv("VITE_KIMI_API_KEY") or os.getenv("KIMI_API_KEY")
base_url = os.getenv("VITE_KIMI_BASE_URL", "https://api.moonshot.cn/v1")

client = OpenAI(api_key=api_key, base_url=base_url)
print("✅ Kimi 客户端初始化成功")

✅ Kimi 客户端初始化成功


## 基础工具调用

In [2]:
# 定义工具
tools = [
    {
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": "获取指定城市的天气信息",
            "parameters": {
                "type": "object",
                "properties": {
                    "city": {
                        "type": "string",
                        "description": "城市名称",
                    },
                    "date": {
                        "type": "string",
                        "description": "日期，格式 YYYY-MM-DD",
                    },
                },
                "required": ["city"],
            },
        },
    }
]

# 发送请求
response = client.chat.completions.create(
    model="kimi-k2-turbo-preview",
    messages=[{
        "role": "user",
        "content": "北京今天天气怎么样？"
    }],
    tools=tools,
    tool_choice="auto",
)

message = response.choices[0].message

if message.tool_calls:
    print("🔧 工具调用请求:")
    for tc in message.tool_calls:
        print(f"  函数名: {tc.function.name}")
        print(f"  参数: {tc.function.arguments}")
else:
    print(f"📄 回答: {message.content}")

🔧 工具调用请求:
  函数名: get_weather
  参数: {"city": "北京"}


## 完整工具调用流程

In [3]:
# 模拟工具函数
def get_weather(city: str, date: str = None):
    """模拟天气查询"""
    return {
        "city": city,
        "weather": "晴朗",
        "temperature": "25°C"
    }

def calculate(expression: str):
    """模拟计算器"""
    try:
        return {"result": eval(expression)}
    except Exception as e:
        return {"error": str(e)}

# 工具映射
tool_functions = {
    "get_weather": get_weather,
    "calculate": calculate,
}

# 工具定义
tools = [
    {
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": "获取天气信息",
            "parameters": {
                "type": "object",
                "properties": {
                    "city": {"type": "string"},
                    "date": {"type": "string"}
                },
                "required": ["city"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "calculate",
            "description": "执行数学计算",
            "parameters": {
                "type": "object",
                "properties": {
                    "expression": {"type": "string"}
                },
                "required": ["expression"]
            }
        }
    }
]

# 对话流程
messages = [
    {"role": "user", "content": "北京天气怎么样？"}
]

# 第一轮：获取工具调用请求
response = client.chat.completions.create(
    model="kimi-k2-turbo-preview",
    messages=messages,
    tools=tools,
)

choice = response.choices[0]

if choice.finish_reason == "tool_calls":
    # 添加助手消息
    messages.append({
        "role": "assistant",
        "content": choice.message.content or "",
        "tool_calls": [
            {
                "id": tc.id,
                "type": tc.type,
                "function": {
                    "name": tc.function.name,
                    "arguments": tc.function.arguments
                }
            } for tc in choice.message.tool_calls
        ]
    })
    
    # 执行工具
    for tc in choice.message.tool_calls:
        func_name = tc.function.name
        func_args = json.loads(tc.function.arguments)
        
        print(f"🔧 执行工具: {func_name}")
        print(f"   参数: {func_args}")
        
        if func_name in tool_functions:
            result = tool_functions[func_name](**func_args)
        else:
            result = {"error": f"Unknown function: {func_name}"}
        
        print(f"   结果: {result}")
        
        # 添加工具结果
        messages.append({
            "role": "tool",
            "tool_call_id": tc.id,
            "name": func_name,
            "content": json.dumps(result)
        })
    
    # 第二轮：获取最终回复
    final_response = client.chat.completions.create(
        model="kimi-k2-turbo-preview",
        messages=messages,
        tools=tools,
    )
    
    print(f"\n📄 最终回复: {final_response.choices[0].message.content}")

🔧 执行工具: get_weather
   参数: {'city': '北京'}
   结果: {'city': '北京', 'weather': '晴朗', 'temperature': '25°C'}

📄 最终回复: 北京今天天气晴朗，温度 25°C。


## 多工具调用

In [4]:
# 测试一次请求中的多个工具调用
response = client.chat.completions.create(
    model="kimi-k2-turbo-preview",
    messages=[{
        "role": "user",
        "content": "北京天气怎么样？15乘以23等于多少？"
    }],
    tools=tools,
)

if response.choices[0].message.tool_calls:
    tool_calls = response.choices[0].message.tool_calls
    print(f"工具调用数量: {len(tool_calls)}")
    
    for i, tc in enumerate(tool_calls, 1):
        print(f"\n调用 {i}:")
        print(f"  函数: {tc.function.name}")
        print(f"  参数: {tc.function.arguments}")

工具调用数量: 2

调用 1:
  函数: get_weather
  参数: {'city': '北京'}

调用 2:
  函数: calculate
  参数: {'expression': '15 * 23'}


## 流式工具调用

In [5]:
# 流式工具调用
response = client.chat.completions.create(
    model="kimi-k2-turbo-preview",
    messages=[{"role": "user", "content": "北京天气如何？"}],
    tools=tools,
    stream=True,
)

tool_calls_acc = []

for chunk in response:
    delta = chunk.choices[0].delta
    
    if delta.tool_calls:
        for tc in delta.tool_calls:
            idx = tc.index
            while len(tool_calls_acc) <= idx:
                tool_calls_acc.append({"id": "", "function": {"name": "", "arguments": ""}})
            
            if tc.id:
                tool_calls_acc[idx]["id"] = tc.id
            if tc.function and tc.function.name:
                tool_calls_acc[idx]["function"]["name"] = tc.function.name
            if tc.function and tc.function.arguments:
                tool_calls_acc[idx]["function"]["arguments"] += tc.function.arguments

print("累积的工具调用:")
for tc in tool_calls_acc:
    print(f"  {tc['function']['name']}: {tc['function']['arguments']}")

累积的工具调用:
  get_weather: {"city": "北京"}
